# Notebook 72R1: Self-contained V8.8 Colab

This combines the operational notebook and `v8_8_operator.py` into one file. No repository clone, external `.py` upload, or path manipulation is required.

**Run all.** Edit only the configuration cell. The frozen operator still requires an observed page, authoritative M3 mask, and either a SAFE_REMOVE image or decision NPZ.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q numpy pandas opencv-python-headless scipy


## Embedded frozen V8.8 operator

In [ ]:
#!/usr/bin/env python3
"""V8.8 registered-replay correction operator.

This operator extracts the stable execution contract from the V8.8 notebook chain:

    M5_REGISTERED_REPLAY = M3 & ~SAFE_REMOVE & ~SAFE_PROTECT

The corrected page is Telea reconstruction inside M5 and a background-paper estimate
at SAFE_REMOVE coordinates. SAFE_PROTECT is supported only for exact historical replay
and restores the observed byte. It is disabled by default.

The script does not train, recalibrate, choose thresholds, create annotation masks, or
claim automatic-output authority. An authoritative reviewed M3 mask and frozen decision
maps are required.
"""

from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import cv2
import numpy as np

SUPPORTED_IMAGES = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}


def require(condition: bool, message: str) -> None:
    if not bool(condition):
        raise RuntimeError(message)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def array_hash(array: np.ndarray) -> str:
    value = np.ascontiguousarray(array)
    digest = hashlib.sha256()
    digest.update(str(value.dtype).encode())
    digest.update(str(value.shape).encode())
    digest.update(value.tobytes())
    return digest.hexdigest()


def read_gray(path: Path) -> np.ndarray:
    require(path.exists(), f"Missing image: {path}")
    require(path.suffix.lower() in SUPPORTED_IMAGES, f"Unsupported image type: {path}")
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f"Unreadable image: {path}")
    return image


def read_mask(path: Path, expected_shape: tuple[int, int], label: str) -> np.ndarray:
    image = read_gray(path)
    require(image.shape == expected_shape, f"{label} shape {image.shape} != observed {expected_shape}")
    return image > 0


def padded_components(mask: np.ndarray, margin: int = 48, minimum_area: int = 8):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    height, width = mask.shape
    boxes = []
    for component in range(1, count):
        x, y, box_width, box_height, area = stats[component]
        if area >= minimum_area:
            boxes.append((
                max(0, x - margin), max(0, y - margin),
                min(width, x + box_width + margin), min(height, y + box_height + margin),
            ))

    changed = True
    while changed:
        changed = False
        merged = []
        while boxes:
            ax0, ay0, ax1, ay1 = boxes.pop()
            rest = []
            for bx0, by0, bx1, by1 in boxes:
                overlaps = not (ax1 < bx0 or bx1 < ax0 or ay1 < by0 or by1 < ay0)
                if overlaps:
                    ax0, ay0 = min(ax0, bx0), min(ay0, by0)
                    ax1, ay1 = max(ax1, bx1), max(ay1, by1)
                    changed = True
                else:
                    rest.append((bx0, by0, bx1, by1))
            boxes = rest
            merged.append((ax0, ay0, ax1, ay1))
        boxes = merged
    return boxes


def background_surface(observed: np.ndarray, authority: np.ndarray, sigma: float = 45.0) -> np.ndarray:
    visible = (~authority) & (observed >= 200)
    values = observed.astype(np.float32) * visible
    weights = visible.astype(np.float32)
    numerator = cv2.GaussianBlur(values, (0, 0), sigma)
    denominator = cv2.GaussianBlur(weights, (0, 0), sigma)
    fallback = cv2.GaussianBlur(observed, (0, 0), sigma)
    surface = np.where(denominator > 0.02, numerator / np.maximum(denominator, 1e-4), fallback)
    return np.clip(surface, 0, 255).astype(np.uint8)


def reconstruct_residual(
    observed: np.ndarray,
    residual: np.ndarray,
    method: str = "telea",
    radius: int = 3,
    crop_margin: int = 48,
) -> np.ndarray:
    result = observed.copy()
    mask_u8 = residual.astype(np.uint8) * 255
    flag = cv2.INPAINT_TELEA if method == "telea" else cv2.INPAINT_NS
    for x0, y0, x1, y1 in padded_components(residual, margin=crop_margin):
        crop = observed[y0:y1, x0:x1]
        crop_mask = mask_u8[y0:y1, x0:x1]
        candidate = cv2.inpaint(crop, crop_mask, radius, flag)
        local = result[y0:y1, x0:x1]
        local[crop_mask > 0] = candidate[crop_mask > 0]
        result[y0:y1, x0:x1] = local
    require(np.array_equal(result[~residual], observed[~residual]),
            "Residual reconstruction changed a non-residual coordinate")
    return result


def load_decisions(
    shape: tuple[int, int],
    safe_remove_path: Path | None,
    safe_protect_path: Path | None,
    decision_npz: Path | None,
    remove_code: int,
    protect_code: int,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    provenance: dict[str, Any] = {}
    if decision_npz is not None:
        require(decision_npz.exists(), f"Missing decision NPZ: {decision_npz}")
        with np.load(decision_npz, allow_pickle=False) as archive:
            if "m5_decision_map" in archive.files:
                decision = np.asarray(archive["m5_decision_map"])
                require(decision.ndim == 2, "m5_decision_map must be two-dimensional")
                require(decision.shape[0] <= shape[0] and decision.shape[1] <= shape[1],
                        "Decision map exceeds page dimensions")
                padded = np.zeros(shape, dtype=decision.dtype)
                padded[: decision.shape[0], : decision.shape[1]] = decision
                safe_remove = padded == remove_code
                safe_protect = padded == protect_code
                provenance["decision_key"] = "m5_decision_map"
                provenance["decision_values"] = np.unique(decision).tolist()
            elif {"SAFE_REMOVE", "M5_REMOVE"}.intersection(archive.files):
                key = "SAFE_REMOVE" if "SAFE_REMOVE" in archive.files else "safe_remove"
                require(key in archive.files, f"No SAFE_REMOVE array in {decision_npz}; keys={archive.files}")
                safe_remove = np.asarray(archive[key]) > 0
                require(safe_remove.shape == shape, "SAFE_REMOVE array shape mismatch")
                safe_protect = np.zeros(shape, bool)
                provenance["decision_key"] = key
            else:
                raise RuntimeError(f"No recognised decision array in {decision_npz}; keys={archive.files}")
        provenance["decision_npz"] = str(decision_npz)
        provenance["decision_npz_sha256"] = sha256_file(decision_npz)
        return safe_remove, safe_protect, provenance

    require(safe_remove_path is not None, "Supply --safe-remove or --decision-npz")
    safe_remove = read_mask(safe_remove_path, shape, "SAFE_REMOVE")
    safe_protect = (
        read_mask(safe_protect_path, shape, "SAFE_PROTECT")
        if safe_protect_path is not None
        else np.zeros(shape, bool)
    )
    provenance.update({
        "safe_remove_path": str(safe_remove_path),
        "safe_remove_sha256": sha256_file(safe_remove_path),
        "safe_protect_path": str(safe_protect_path) if safe_protect_path else None,
        "safe_protect_sha256": sha256_file(safe_protect_path) if safe_protect_path else None,
    })
    return safe_remove, safe_protect, provenance


def write_image(path: Path, image: np.ndarray) -> None:
    require(cv2.imwrite(str(path), image), f"Could not write image: {path}")


def run(args: argparse.Namespace) -> Path:
    observed_path = Path(args.observed)
    m3_path = Path(args.m3)
    output_dir = Path(args.output)
    require(not output_dir.exists() or not any(output_dir.iterdir()),
            f"Output directory must not contain files: {output_dir}")
    output_dir.mkdir(parents=True, exist_ok=True)

    observed = read_gray(observed_path)
    m3 = read_mask(m3_path, observed.shape, "M3")
    safe_remove, safe_protect, decision_provenance = load_decisions(
        observed.shape,
        Path(args.safe_remove) if args.safe_remove else None,
        Path(args.safe_protect) if args.safe_protect else None,
        Path(args.decision_npz) if args.decision_npz else None,
        args.remove_code,
        args.protect_code,
    )

    require(not np.any(safe_remove & ~m3), "SAFE_REMOVE is not a subset of M3")
    require(not np.any(safe_protect & ~m3), "SAFE_PROTECT is not a subset of M3")
    require(not np.any(safe_remove & safe_protect), "SAFE_REMOVE and SAFE_PROTECT overlap")
    if safe_protect.any() and not args.enable_safe_protect_replay:
        raise RuntimeError(
            "SAFE_PROTECT decisions are present. They are disabled operationally. "
            "Use --enable-safe-protect-replay only for exact historical registered replay."
        )

    m5 = m3 & ~safe_remove & ~safe_protect
    residual_reconstruction = reconstruct_residual(
        observed, m5, method=args.method, radius=args.radius, crop_margin=args.crop_margin
    )
    paper = background_surface(observed, m3, sigma=args.paper_sigma)
    corrected = residual_reconstruction.copy()
    corrected[safe_remove] = paper[safe_remove]
    corrected[safe_protect] = observed[safe_protect]

    outside_drift = int(np.sum(corrected[~m3] != observed[~m3]))
    m5_formula_xor = int(np.logical_xor(m5, m3 & ~safe_remove & ~safe_protect).sum())
    protect_drift = int(np.sum(corrected[safe_protect] != observed[safe_protect]))
    require(outside_drift == 0, f"Outside-M3 immutability failure: {outside_drift} pixels")
    require(m5_formula_xor == 0, f"M5 formula failure: {m5_formula_xor} pixels")
    require(protect_drift == 0, f"SAFE_PROTECT exact-copy failure: {protect_drift} pixels")

    write_image(output_dir / "observed.png", observed)
    write_image(output_dir / "m3.png", m3.astype(np.uint8) * 255)
    write_image(output_dir / "safe_remove.png", safe_remove.astype(np.uint8) * 255)
    write_image(output_dir / "safe_protect_decision.png", safe_protect.astype(np.uint8) * 255)
    write_image(output_dir / "m5.png", m5.astype(np.uint8) * 255)
    write_image(output_dir / "residual_reconstruction.png", residual_reconstruction)
    write_image(output_dir / "paper_estimate.png", paper)
    write_image(output_dir / "corrected.png", corrected)
    difference = np.zeros_like(observed)
    difference[corrected != observed] = 255
    write_image(output_dir / "difference_from_observed.png", difference)
    np.savez_compressed(
        output_dir / "v8_8_masks.npz",
        M3=m3.astype(np.uint8),
        SAFE_REMOVE=safe_remove.astype(np.uint8),
        SAFE_PROTECT=safe_protect.astype(np.uint8),
        M5=m5.astype(np.uint8),
    )

    manifest = {
        "schema": "music-annotation-v8.8-operator-run-v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "operator_version": "8.8-registered-replay-operator-1.0",
        "contract": "M5 = M3 & ~SAFE_REMOVE & ~SAFE_PROTECT",
        "observed": str(observed_path),
        "observed_sha256": sha256_file(observed_path),
        "m3": str(m3_path),
        "m3_sha256": sha256_file(m3_path),
        "decision_provenance": decision_provenance,
        "method": args.method,
        "inpaint_radius": args.radius,
        "crop_margin": args.crop_margin,
        "paper_sigma": args.paper_sigma,
        "safe_protect_replay_enabled": bool(args.enable_safe_protect_replay),
        "automatic_annotation_mask_authority": False,
        "automatic_output_authority": False,
        "human_approval_required": True,
        "counts": {
            "m3_pixels": int(m3.sum()),
            "safe_remove_pixels": int(safe_remove.sum()),
            "safe_protect_pixels": int(safe_protect.sum()),
            "m5_pixels": int(m5.sum()),
            "changed_pixels": int((corrected != observed).sum()),
        },
        "integrity": {
            "outside_m3_drift": outside_drift,
            "m5_formula_xor": m5_formula_xor,
            "safe_protect_copy_drift": protect_drift,
        },
        "semantic_hashes": {
            "M3": array_hash(m3.astype(np.uint8)),
            "SAFE_REMOVE": array_hash(safe_remove.astype(np.uint8)),
            "SAFE_PROTECT": array_hash(safe_protect.astype(np.uint8)),
            "M5": array_hash(m5.astype(np.uint8)),
            "corrected": array_hash(corrected),
        },
        "software": {
            "python": sys.version,
            "platform": platform.platform(),
            "opencv": cv2.__version__,
            "numpy": np.__version__,
        },
    }
    (output_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True))
    return output_dir


def parser() -> argparse.ArgumentParser:
    value = argparse.ArgumentParser(
        description="Build a V8.8 registered-replay corrected page from observed, M3 and frozen decisions."
    )
    value.add_argument("--observed", required=True, help="Observed/aligned grayscale page")
    value.add_argument("--m3", required=True, help="Authoritative reviewed M3 intervention mask")
    decision = value.add_mutually_exclusive_group(required=True)
    decision.add_argument("--safe-remove", help="Binary SAFE_REMOVE image")
    decision.add_argument("--decision-npz", help="NPZ containing m5_decision_map or SAFE_REMOVE")
    value.add_argument("--safe-protect", help="Optional binary SAFE_PROTECT image for historical replay")
    value.add_argument("--enable-safe-protect-replay", action="store_true",
                       help="Permit historical SAFE_PROTECT exact-copy replay; disabled by default")
    value.add_argument("--remove-code", type=int, default=1, help="SAFE_REMOVE decision-map code")
    value.add_argument("--protect-code", type=int, default=2, help="SAFE_PROTECT decision-map code")
    value.add_argument("--method", choices=["telea", "navier_stokes"], default="telea")
    value.add_argument("--radius", type=int, default=3)
    value.add_argument("--crop-margin", type=int, default=48)
    value.add_argument("--paper-sigma", type=float, default=45.0)
    value.add_argument("--output", required=True, help="New or empty output directory")
    return value


def main() -> int:
    try:
        output = run(parser().parse_args())
        print(f"V8.8 operator completed: {output}")
        print("Human approval remains required; automatic annotation-mask authority is not enabled.")
        return 0
    except Exception as error:
        print(f"V8.8 operator failed closed: {error}", file=sys.stderr)
        return 2


if __name__ == "__main__":
    raise SystemExit(main())


## Configuration
Set the three input locations. For an NPZ decision map, set `SAFE_REMOVE = None` and provide `DECISION_NPZ`. Output must be new or empty.

In [ ]:
from pathlib import Path
import argparse, json

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
OBSERVED = ROOT / 'source/annotated_only/Page001.png'
M3 = ROOT / 'masks/corrected/Page001.png'
SAFE_REMOVE = ROOT / 'v8_8_operational_inputs/safe_remove/Page001.png'
DECISION_NPZ = None
OUTPUT = ROOT / 'notebook72r1_outputs/Page001'
METHOD = 'telea'
RADIUS = 3
CROP_MARGIN = 48
PAPER_SIGMA = 45.0

print('Observed:', OBSERVED)
print('M3:', M3)
print('SAFE_REMOVE:', SAFE_REMOVE)
print('Output:', OUTPUT)


## Run the frozen operator

In [ ]:
args = argparse.Namespace(
    observed=str(OBSERVED),
    m3=str(M3),
    safe_remove=str(SAFE_REMOVE) if SAFE_REMOVE is not None else None,
    decision_npz=str(DECISION_NPZ) if DECISION_NPZ is not None else None,
    safe_protect=None,
    enable_safe_protect_replay=False,
    remove_code=1,
    protect_code=2,
    method=METHOD,
    radius=RADIUS,
    crop_margin=CROP_MARGIN,
    paper_sigma=PAPER_SIGMA,
    output=str(OUTPUT),
)
result_dir = run(args)
print('Completed:', result_dir)
print((Path(result_dir) / 'audit' / 'run_summary.json').read_text())


## Preview

In [ ]:
import cv2, matplotlib.pyplot as plt
from pathlib import Path
files = [
    ('Observed', Path(result_dir)/'pages'/'observed.png'),
    ('M3', Path(result_dir)/'masks'/'m3.png'),
    ('SAFE_REMOVE', Path(result_dir)/'masks'/'safe_remove.png'),
    ('M5_REMOVE', Path(result_dir)/'masks'/'m5_remove.png'),
    ('Corrected', Path(result_dir)/'pages'/'corrected.png'),
]
fig, axes = plt.subplots(1, len(files), figsize=(22,7))
for ax,(title,path) in zip(axes,files):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    ax.imshow(im,cmap='gray',vmin=0,vmax=255); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()


## What this deliberately does not hide

This is easy to run because the code is in one notebook. It does **not** invent the missing frozen SAFE_REMOVE scorer. New pages still need SAFE_REMOVE decisions from the replay-verified deployment bundle. That constraint is part of the V8.8 safety contract, not notebook complexity.
